## Colab Setup

In [1]:
import os

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/ImmunoPath"
DATA_DIR = f"{PROJECT_DIR}/data"

# Create all data directories
for d in [
    f"{DATA_DIR}/raw/slides",
    f"{DATA_DIR}/raw/rnaseq",
    f"{DATA_DIR}/raw/manifests",
    f"{DATA_DIR}/labels/bagaev_tme",
    f"{DATA_DIR}/labels/saltz_til",
    f"{DATA_DIR}/labels/msi_status",
    f"{DATA_DIR}/labels/thorsson_panimmune",
    f"{DATA_DIR}/labels/cbioportal",
]:
    os.makedirs(d, exist_ok=True)

print(f"Data directories created under: {DATA_DIR}")


Mounted at /content/drive
Data directories created under: /content/drive/MyDrive/ImmunoPath/data


## Install Dependencies

In [2]:
import subprocess
subprocess.run(["pip", "install", "-q", "requests", "pandas", "tqdm"], check=True)

import requests
import json
import pandas as pd
from tqdm.auto import tqdm
import time
import subprocess

print("Dependencies ready")

Dependencies ready


## Query GDC for TCGA-LUAD/LUSC Diagnostic Slides

In [3]:
GDC_FILES_ENDPOINT = "https://api.gdc.cancer.gov/files"
GDC_DATA_ENDPOINT = "https://api.gdc.cancer.gov/data"
PROJECTS = ["TCGA-LUAD", "TCGA-LUSC"]  # Primary lung cancer types

def query_gdc_slides(project_ids):
    """Query GDC API for diagnostic slide file metadata."""
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {
                "field": "cases.project.project_id",
                "value": project_ids
            }},
            {"op": "=", "content": {
                "field": "data_type",
                "value": "Slide Image"
            }},
            {"op": "=", "content": {
                "field": "experimental_strategy",
                "value": "Diagnostic Slide"
            }},
        ]
    }

    params = {
        "filters": json.dumps(filters),
        "fields": ",".join([
            "file_id", "file_name", "file_size",
            "cases.case_id", "cases.submitter_id",
            "cases.project.project_id",
            "cases.samples.sample_type",
        ]),
        "format": "JSON",
        "size": "10000",
    }

    print(f"Querying GDC for diagnostic slides in {project_ids}...")
    response = requests.get(GDC_FILES_ENDPOINT, params=params)
    response.raise_for_status()

    data = response.json()["data"]
    hits = data["hits"]
    total = data["pagination"]["total"]
    print(f"  Found {total} diagnostic slides ({len(hits)} returned)")
    return hits

slide_metadata = query_gdc_slides(PROJECTS)

# Parse into a clean DataFrame
slide_records = []
for hit in slide_metadata:
    case_info = hit.get("cases", [{}])[0]
    slide_records.append({
        "file_id": hit["file_id"],
        "file_name": hit["file_name"],
        "file_size_gb": hit["file_size"] / 1e9,
        "case_id": case_info.get("case_id", ""),
        "submitter_id": case_info.get("submitter_id", ""),
        "project_id": case_info.get("project", {}).get("project_id", ""),
    })

slides_df = pd.DataFrame(slide_records)
print(f"\nSlide Summary:")
print(slides_df["project_id"].value_counts().to_string())
print(f"\nTotal size: {slides_df['file_size_gb'].sum():.1f} GB")
print(f"Unique patients: {slides_df['submitter_id'].nunique()}")

# Save metadata
slides_csv = f"{DATA_DIR}/raw/manifests/slide_metadata.csv"
slides_df.to_csv(slides_csv, index=False)
print(f"\nSlide metadata saved to: {slides_csv}")


Querying GDC for diagnostic slides in ['TCGA-LUAD', 'TCGA-LUSC']...
  Found 1053 diagnostic slides (1053 returned)

Slide Summary:
project_id
TCGA-LUAD    541
TCGA-LUSC    512

Total size: 823.7 GB
Unique patients: 956

Slide metadata saved to: /content/drive/MyDrive/ImmunoPath/data/raw/manifests/slide_metadata.csv


## Query GDC for RNA-seq (STAR - Counts)

In [ ]:
# We need gene expression data to compute immune signatures.

def query_gdc_rnaseq(project_ids):
    """Query GDC API for RNA-seq gene expression quantification files."""
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {
                "field": "cases.project.project_id",
                "value": project_ids
            }},
            {"op": "=", "content": {
                "field": "data_type",
                "value": "Gene Expression Quantification"
            }},
            {"op": "=", "content": {
                "field": "analysis.workflow_type",
                "value": "STAR - Counts"
            }},
            # Primary tumor samples only
            {"op": "=", "content": {
                "field": "cases.samples.sample_type",
                "value": "Primary Tumor"
            }},
        ]
    }

    params = {
        "filters": json.dumps(filters),
        "fields": ",".join([
            "file_id", "file_name", "file_size",
            "cases.case_id", "cases.submitter_id",
            "cases.project.project_id",
            "cases.samples.submitter_id",
        ]),
        "format": "JSON",
        "size": "10000",
    }

    print(f"Querying GDC for RNA-seq (STAR - Counts) in {project_ids}...")
    response = requests.get(GDC_FILES_ENDPOINT, params=params)
    response.raise_for_status()

    data = response.json()["data"]
    hits = data["hits"]
    total = data["pagination"]["total"]
    print(f"  Found {total} RNA-seq files ({len(hits)} returned)")
    return hits

rnaseq_metadata = query_gdc_rnaseq(PROJECTS)

# Parse into DataFrame
rnaseq_records = []
for hit in rnaseq_metadata:
    case_info = hit.get("cases", [{}])[0]
    rnaseq_records.append({
        "file_id": hit["file_id"],
        "file_name": hit["file_name"],
        "file_size_mb": hit["file_size"] / 1e6,
        "case_id": case_info.get("case_id", ""),
        "submitter_id": case_info.get("submitter_id", ""),
        "project_id": case_info.get("project", {}).get("project_id", ""),
    })

rnaseq_df = pd.DataFrame(rnaseq_records)
print(f"\nRNA-seq Summary:")
print(rnaseq_df["project_id"].value_counts().to_string())
print(f"Unique patients: {rnaseq_df['submitter_id'].nunique()}")

# Save metadata
rnaseq_csv = f"{DATA_DIR}/raw/manifests/rnaseq_metadata.csv"
rnaseq_df.to_csv(rnaseq_csv, index=False)
print(f"\nRNA-seq metadata saved to: {rnaseq_csv}")

Querying GDC for RNA-seq (STAR - Counts) in ['TCGA-LUAD', 'TCGA-LUSC']...
  Found 1051 RNA-seq files (1051 returned)

RNA-seq Summary:
project_id
TCGA-LUAD    540
TCGA-LUSC    511
Unique patients: 1018

RNA-seq metadata saved to: /content/drive/MyDrive/ImmunoPath/data/raw/manifests/rnaseq_metadata.csv


## Find matched patients(RNA+Seq)

In [5]:
slide_patients = set(slides_df["submitter_id"])
rnaseq_patients = set(rnaseq_df["submitter_id"])
matched_patients = slide_patients & rnaseq_patients

print(f"Patients with slides:    {len(slide_patients)}")
print(f"Patients with RNA-seq:   {len(rnaseq_patients)}")
print(f"Matched (both):          {len(matched_patients)}")
print(f"Slides only:             {len(slide_patients - rnaseq_patients)}")
print(f"RNA-seq only:            {len(rnaseq_patients - slide_patients)}")

# Filter to matched patients
matched_slides = slides_df[slides_df["submitter_id"].isin(matched_patients)]
matched_rnaseq = rnaseq_df[rnaseq_df["submitter_id"].isin(matched_patients)]

# If patient has multiple slides, keep one per patient (prefer smallest for download speed)
matched_slides = (matched_slides
    .sort_values("file_size_gb")
    .drop_duplicates(subset="submitter_id", keep="first"))

# Same for RNA-seq
matched_rnaseq = (matched_rnaseq
    .sort_values("file_size_mb")
    .drop_duplicates(subset="submitter_id", keep="first"))

print(f"\nAfter dedup (1 per patient):")
print(f"  Slides:  {len(matched_slides)}")
print(f"  RNA-seq: {len(matched_rnaseq)}")
print(f"  Total slide download: {matched_slides['file_size_gb'].sum():.1f} GB")

# Save the matched manifest
nsclc_metadata = matched_slides[["file_id", "file_name", "submitter_id", "project_id", "file_size_gb"]]
nsclc_metadata.to_csv(f"{DATA_DIR}/raw/nsclc_metadata.csv", index=False)
print(f"\nMatched metadata saved")


Patients with slides:    956
Patients with RNA-seq:   1018
Matched (both):          950
Slides only:             6
RNA-seq only:            68

After dedup (1 per patient):
  Slides:  950
  RNA-seq: 950
  Total slide download: 718.9 GB

Matched metadata saved


## Download Slides via GDC API

In [ ]:
import os, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_SLIDES = 950
MAX_WORKERS = 4
CHUNK_SIZE = 1024 * 1024  # 1 MB chunks
TIMEOUT = 120  # seconds per request

# Session with retries + connection pooling (same pattern as RNA-seq)
slide_session = requests.Session()
slide_adapter = requests.adapters.HTTPAdapter(
    pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS, max_retries=3
)
slide_session.mount("https://", slide_adapter)

# Stage to local SSD first, then move to GDrive (avoids slow FUSE writes)
LOCAL_SLIDE_TMP = "/content/slide_tmp"
os.makedirs(LOCAL_SLIDE_TMP, exist_ok=True)

def download_gdc_slide(row):
    """Download a single slide with retry, timeout, resume-safe atomic write."""
    file_id = row["file_id"]
    filename = row["file_name"]
    submitter_id = row["submitter_id"]
    final_path = os.path.join(f"{DATA_DIR}/raw/slides", filename)

    # Skip if already fully downloaded (check file size vs expected)
    if os.path.exists(final_path):
        actual_size = os.path.getsize(final_path)
        expected_size = int(row["file_size_gb"] * 1e9)
        if actual_size >= expected_size * 0.95:  # within 5% tolerance
            return submitter_id
        else:
            os.remove(final_path)  # partial download — remove and retry

    url = f"{GDC_DATA_ENDPOINT}/{file_id}"
    tmp_path = os.path.join(LOCAL_SLIDE_TMP, f"{file_id}.tmp")

    with slide_session.get(url, stream=True, timeout=TIMEOUT) as r:
        r.raise_for_status()
        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)

    # Atomic move: local tmp → GDrive final path
    shutil.move(tmp_path, final_path)
    return submitter_id

slides_to_download = matched_slides.head(MAX_SLIDES)
print(f"Downloading {len(slides_to_download)} slides ({MAX_WORKERS} parallel workers)")
print(f"Estimated size: {slides_to_download['file_size_gb'].sum():.1f} GB")
print(f"Staging via local SSD → GDrive for faster I/O")
print()

slide_dir = f"{DATA_DIR}/raw/slides"
os.makedirs(slide_dir, exist_ok=True)
downloaded_slides = []
failed_slides = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_gdc_slide, row): row
        for _, row in slides_to_download.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading slides"):
        row = futures[future]
        try:
            sid = future.result()
            downloaded_slides.append(sid)
        except Exception as e:
            failed_slides.append(row["file_name"])
            print(f"Failed: {row['file_name']}: {e}")

print(f"\nDownloaded {len(downloaded_slides)}/{len(slides_to_download)} slides")
if failed_slides:
    print(f"Failed ({len(failed_slides)}): {failed_slides[:5]}...")


Estimated size: 0.4 GB
Output dir: /content/drive/MyDrive/ImmunoPath/data/raw/slides/



TCGA-05-4425-01Z-00-DX1.82B093:   0%|          | 0.00/6.38M [00:00<?, ?B/s]

TCGA-05-5715-01Z-00-DX1.D3F0A1:   0%|          | 0.00/7.96M [00:00<?, ?B/s]

TCGA-05-5429-01Z-00-DX1.207290:   0%|          | 0.00/8.74M [00:00<?, ?B/s]

TCGA-55-8207-01Z-00-DX1.2dafc4:   0%|          | 0.00/8.77M [00:00<?, ?B/s]

...

TCGA-55-8505-01Z-00-DX1.D364C3:   0%|          | 0.00/37.6M [00:00<?, ?B/s]

TCGA-98-8022-01Z-00-DX1.870ca0:   0%|          | 0.00/37.8M [00:00<?, ?B/s]

TCGA-56-5898-01Z-00-DX1.d539fc:   0%|          | 0.00/38.8M [00:00<?, ?B/s]

TCGA-99-7458-01Z-00-DX1.10ea0b:   0%|          | 0.00/49.9M [00:00<?, ?B/s]


Downloaded 20/20 slides


## Download RNA-seq Files

In [8]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Config
GDC_DATA_ENDPOINT = "https://api.gdc.cancer.gov/data"
rnaseq_dir = f"{DATA_DIR}/raw/rnaseq"
os.makedirs(rnaseq_dir, exist_ok=True)

MAX_WORKERS = 16
CHUNK_SIZE = 1024 * 1024  # 1 MB
TIMEOUT = 60

session = requests.Session()
adapter = requests.adapters.HTTPAdapter(pool_connections=MAX_WORKERS,
                                         pool_maxsize=MAX_WORKERS,
                                         max_retries=3)
session.mount("https://", adapter)

def download_rnaseq(row):
    file_id = row["file_id"]
    submitter_id = row["submitter_id"]
    output_path = os.path.join(rnaseq_dir, f"{submitter_id}.tsv.gz")

    if os.path.exists(output_path):
        return submitter_id

    url = f"{GDC_DATA_ENDPOINT}/{file_id}"
    with session.get(url, stream=True, timeout=TIMEOUT) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)

    return submitter_id

print(f"Downloading {len(matched_rnaseq)} RNA-seq files...")

downloaded = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(download_rnaseq, row)
               for _, row in matched_rnaseq.iterrows()]

    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            downloaded.append(future.result())
        except Exception as e:
            print(f"Failed: {e}")

print(f"\nDownloaded {len(downloaded)}/{len(matched_rnaseq)} RNA-seq files")


 57%|█████▋    | 539/950 [01:40<01:18,  5.25it/s]

Failed: 500 Server Error: Internal Server Error for url: https://api.gdc.cancer.gov/data/ddf190fa-393b-46f9-8793-c6eb4f08313b
Failed: 500 Server Error: Internal Server Error for url: https://api.gdc.cancer.gov/data/475fdbbb-3d3a-4d56-8d0e-34dfb8cb6346


100%|██████████| 950/950 [02:55<00:00,  5.42it/s]


Downloaded 948/950 RNA-seq files


## Download Bagaev TME Subtype Labels

In [10]:
import os
import requests

BOSTONGENE_BASE = "https://raw.githubusercontent.com/BostonGene/MFP/master"
bagaev_dir = f"{DATA_DIR}/labels/bagaev_tme"
sig_dir = os.path.join(bagaev_dir, "signatures")

# ensure directories exist
os.makedirs(bagaev_dir, exist_ok=True)
os.makedirs(sig_dir, exist_ok=True)

# files to download
files = {
    "annotation.tsv": f"{BOSTONGENE_BASE}/Cohorts/Pan_TCGA/annotation.tsv",
    "signatures.tsv": f"{BOSTONGENE_BASE}/Cohorts/Pan_TCGA/signatures.tsv",
    # correct gene signature GMT
    "gene_signatures.gmt": f"{BOSTONGENE_BASE}/signatures/gene_signatures.gmt",
}

for filename, url in files.items():
    output_path = os.path.join(sig_dir if filename.endswith(".gmt") else bagaev_dir, filename)
    print(f"Downloading {filename} from {url}...")
    try:
        response = requests.get(url)
        response.raise_for_status()
        with open(output_path, "w") as f:
            f.write(response.text)
        print(f"Saved {filename} → {output_path}")
    except Exception as e:
        print(f"Failed to download {filename}: {e}")


Saved annotation.tsv → /content/drive/MyDrive/ImmunoPath/data/labels/bagaev_tme/annotation.tsv
Saved signatures.tsv → /content/drive/MyDrive/ImmunoPath/data/labels/bagaev_tme/signatures.tsv
Saved gene_signatures.gmt → /content/drive/MyDrive/ImmunoPath/data/labels/bagaev_tme/signatures/gene_signatures.gmt


## Download MSI Status + Clinical Data from cBioPortal

In [ ]:
# Source: cBioPortal TCGA PanCancer Atlas
CBIOPORTAL_STUDIES = {
    "TCGA-LUAD": "https://datahub.assets.cbioportal.org/luad_tcga_pan_can_atlas_2018.tar.gz",
    "TCGA-LUSC": "https://datahub.assets.cbioportal.org/lusc_tcga_pan_can_atlas_2018.tar.gz",
}

cbio_dir = f"{DATA_DIR}/labels/cbioportal"

for study_name, url in CBIOPORTAL_STUDIES.items():
    archive_name = url.split("/")[-1]
    archive_path = os.path.join(cbio_dir, archive_name)
    extract_dir = os.path.join(cbio_dir, archive_name.replace(".tar.gz", ""))

    if os.path.exists(extract_dir):
        print(f"Already extracted: {study_name}")
        continue

    print(f"\nDownloading {study_name} from cBioPortal...")
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        total = int(response.headers.get('content-length', 0))

        with open(archive_path, 'wb') as f:
            with tqdm(total=total, unit='B', unit_scale=True, desc=study_name) as pbar:
                for chunk in response.iter_content(8192*8):
                    f.write(chunk)
                    pbar.update(len(chunk))

        # Extract
        subprocess.run(["tar", "-xzf", archive_path, "-C", cbio_dir], check=True)
        os.remove(archive_path)  # Clean up archive
        print(f" Extracted to {extract_dir}")
    except Exception as e:
        print(f"Failed: {e}")

# Parse MSI status from cBioPortal clinical data
print("\nExtracting MSI status from cBioPortal data...")
msi_records = []

for study_name in ["luad_tcga_pan_can_atlas_2018", "lusc_tcga_pan_can_atlas_2018"]:
    clinical_file = os.path.join(cbio_dir, study_name, "data_clinical_sample.txt")
    if not os.path.exists(clinical_file):
        # Try alternative path
        clinical_file = os.path.join(cbio_dir, study_name, "data_clinical_patient.txt")
    if not os.path.exists(clinical_file):
        print(f"Clinical file not found for {study_name}")
        continue

    try:
        # cBioPortal clinical files have comment lines starting with #
        df = pd.read_csv(clinical_file, sep='\t', comment='#')
        print(f"  {study_name}: {len(df)} samples, columns: {list(df.columns)[:10]}...")

        # Look for MSI-related columns
        msi_cols = [c for c in df.columns if 'msi' in c.lower() or 'microsatellite' in c.lower()]
        if msi_cols:
            print(f"    MSI columns found: {msi_cols}")
        else:
            print(f"    No MSI columns found (will use Thorsson data instead)")
    except Exception as e:
        print(f"Error parsing {study_name}: {e}")


TCGA-LUAD: 100%|██████████| 285M/285M [00:11<00:00, 24.6MB/s]


 Extracted to /content/drive/MyDrive/ImmunoPath/data/labels/cbioportal/luad_tcga_pan_can_atlas_2018



TCGA-LUSC: 100%|██████████| 257M/257M [00:10<00:00, 24.5MB/s]


 Extracted to /content/drive/MyDrive/ImmunoPath/data/labels/cbioportal/lusc_tcga_pan_can_atlas_2018

Extracting MSI status from cBioPortal data...
  luad_tcga_pan_can_atlas_2018: 566 samples, columns: ['PATIENT_ID', 'SAMPLE_ID', 'ONCOTREE_CODE', 'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'TUMOR_TYPE', 'GRADE', 'TISSUE_PROSPECTIVE_COLLECTION_INDICATOR', 'TISSUE_RETROSPECTIVE_COLLECTION_INDICATOR', 'TISSUE_SOURCE_SITE_CODE']...
    MSI columns found: ['MSI_SCORE_MANTIS', 'MSI_SENSOR_SCORE']
  lusc_tcga_pan_can_atlas_2018: 487 samples, columns: ['PATIENT_ID', 'SAMPLE_ID', 'ONCOTREE_CODE', 'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'TUMOR_TYPE', 'GRADE', 'TISSUE_PROSPECTIVE_COLLECTION_INDICATOR', 'TISSUE_RETROSPECTIVE_COLLECTION_INDICATOR', 'TISSUE_SOURCE_SITE_CODE']...
    MSI columns found: ['MSI_SCORE_MANTIS', 'MSI_SENSOR_SCORE']


## Download Thorsson Panimmune Data from GDC

In [13]:
import os
import time
import tarfile
import requests
import pandas as pd

# ------------------------------------------------------------------
# Thorsson et al. 2018 — Pan-Immune TCGA
# ------------------------------------------------------------------

THORSSON_FILES = {
    "immune_signatures_160.tsv.gz": "https://api.gdc.cancer.gov/data/80a82092-161d-4615-9d96-e858f113618d",
    "leukocyte_fractions.tsv":      "https://api.gdc.cancer.gov/data/6f75c9d7-5134-4ed1-b8f3-72856c98a4e8",
    "cibersort_fractions.tsv":      "https://api.gdc.cancer.gov/data/b3df502e-3594-46ef-9f94-d041a20a0b9a",
    "mutation_load.tsv":            "https://api.gdc.cancer.gov/data/ff3f962c-3573-44ae-a8f4-e5ac0aea64b6",
    "tcga_subtypes.tsv":             "https://api.gdc.cancer.gov/data/0f31b768-7f67-4fc4-abc3-06ac5bd90bf0",
    "gene_set_definitions.tsv":      "https://api.gdc.cancer.gov/data/9b174979-fe97-48bc-9e97-9384b0519f03",
}

# ------------------------------------------------------------------
# Paths (DO NOT CHANGE STRUCTURE)
# ------------------------------------------------------------------

THORSSON_DIR = f"{DATA_DIR}/labels/thorsson_panimmune"
TMP_DIR      = "/content/thorsson_tmp"
ARCHIVE_PATH = f"{THORSSON_DIR}.tar.gz"

os.makedirs(THORSSON_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# ------------------------------------------------------------------
# Step 1: Download to LOCAL VM (no Drive I/O)
# ------------------------------------------------------------------

print("Downloading Thorsson files to local VM...\n")

for fname, url in THORSSON_FILES.items():
    tmp_path = os.path.join(TMP_DIR, fname)

    if os.path.exists(tmp_path):
        print(f"Already downloaded: {fname}")
        continue

    print(f"Downloading {fname}...")
    r = requests.get(url, stream=True)
    r.raise_for_status()

    with open(tmp_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    size_mb = os.path.getsize(tmp_path) / 1e6
    print(f"  {size_mb:.1f} MB")
    time.sleep(1)

# ------------------------------------------------------------------
# Step 2: Archive once (single Drive write)
# ------------------------------------------------------------------

print("\nCreating archive...")
with tarfile.open(ARCHIVE_PATH, "w:gz") as tar:
    tar.add(TMP_DIR, arcname="thorsson_panimmune")

print(f"Archive written → {ARCHIVE_PATH}")

# ------------------------------------------------------------------
# Step 3: Extract into final folder (only once)
# ------------------------------------------------------------------

print("\nExtracting into final directory...")
with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
    tar.extractall(path=f"{DATA_DIR}/labels")

print("Extraction complete.")

# ------------------------------------------------------------------
# Step 4: Quick inspection
# ------------------------------------------------------------------

print("\nThorsson Data Inspection:")
for fname in THORSSON_FILES:
    path = os.path.join(THORSSON_DIR, fname)
    if not os.path.exists(path):
        print(f"  MISSING: {fname}")
        continue

    try:
        if fname.endswith(".gz"):
            df = pd.read_csv(path, sep="\t", compression="gzip", nrows=3)
        else:
            df = pd.read_csv(path, sep="\t", nrows=3)

        print(f"  {fname}: {len(df.columns)} cols")
    except Exception:
        size = os.path.getsize(path) / 1e6
        print(f"  {fname}: {size:.1f} MB (binary)")

print("\nDONE. No Drive quota pain.")



  7.9 MB
  0.6 MB
  3.8 MB
  0.6 MB
  0.6 MB
  0.1 MB

Creating archive...
Archive written → /content/drive/MyDrive/ImmunoPath/data/labels/thorsson_panimmune.tar.gz

Extracting into final directory...


/tmp/ipython-input-2200223560.py:73: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=f"{DATA_DIR}/labels")


Extraction complete.

Thorsson Data Inspection:
  immune_signatures_160.tsv.gz: 9131 cols
  leukocyte_fractions.tsv: 3 cols
  cibersort_fractions.tsv: 27 cols
  mutation_load.tsv: 5 cols
  tcga_subtypes.tsv: 10 cols
  gene_set_definitions.tsv: 0.1 MB (binary)

DONE. No Drive quota pain.


## Download Saltz TIL Map Summary

## Download the three files
Tilmap summary,
Adjusted indices,
TIL Pattern

## Create Unified MSI Labels File

In [ ]:
# Combine MSI information from available sources.
print("Creating unified MSI labels...")

# Approach 1: Use Thorsson mutation load data (has MSI info)
mutation_file = f"{DATA_DIR}/labels/thorsson_panimmune/mutation_load.tsv"
if os.path.exists(mutation_file):
    try:
        mut_df = pd.read_csv(mutation_file, sep='\t')
        print(f"  Thorsson mutation data: {len(mut_df)} rows")
        print(f"  Columns: {list(mut_df.columns)[:10]}")

        # Look for MSI-related columns
        # Common names: 'MSI MANTIS Score', 'MSIsensor Score', etc.
        msi_cols = [c for c in mut_df.columns if 'msi' in c.lower()]
        if msi_cols:
            print(f"  MSI columns: {msi_cols}")
    except Exception as e:
        print(f"Error reading mutation data: {e}")

# Approach 2: Extract from cBioPortal downloads
for study in ["luad_tcga_pan_can_atlas_2018", "lusc_tcga_pan_can_atlas_2018"]:
    mutation_file = os.path.join(cbio_dir, study, "data_mutations.txt")
    clinical_file = os.path.join(cbio_dir, study, "data_clinical_patient.txt")

    for f in [mutation_file, clinical_file]:
        if os.path.exists(f):
            try:
                df = pd.read_csv(f, sep='\t', comment='#', nrows=5)
                msi_cols = [c for c in df.columns if 'msi' in c.lower()]
                if msi_cols:
                    print(f"  Found MSI in {os.path.basename(f)}: {msi_cols}")
            except Exception:
                pass

print("\n📋 NOTE: If MSI labels are not directly available in these downloads,")
print("   they will be derived from Thorsson immune subtypes (C6 = MSI-H).")
print("   Alternatively, use MSIsensor scores from Broad Firehose:")
print("   https://gdac.broadinstitute.org/")

# Save a placeholder MSI file that Phase 2 will populate
msi_output = f"{DATA_DIR}/labels/msi_status.tsv"
if not os.path.exists(msi_output):
    pd.DataFrame(columns=["submitter_id", "msi_status", "msi_score", "source"])\
      .to_csv(msi_output, sep='\t', index=False)
    print(f"\n  Created placeholder: {msi_output}")
    print("  Phase 2 will populate this with actual MSI labels.")

Creating unified MSI labels...
  Thorsson mutation data: 10123 rows
  Columns: ['Cohort', 'Patient_ID', 'Tumor_Sample_ID', 'Silent per Mb', 'Non-silent per Mb']

📋 NOTE: If MSI labels are not directly available in these downloads,
   they will be derived from Thorsson immune subtypes (C6 = MSI-H).
   Alternatively, use MSIsensor scores from Broad Firehose:
   https://gdac.broadinstitute.org/

  Created placeholder: /content/drive/MyDrive/ImmunoPath/data/labels/msi_status.tsv
  Phase 2 will populate this with actual MSI labels.


## Generate Download Summary Report

In [ ]:
import json
from datetime import datetime

report = {
    "phase": 1,
    "timestamp": datetime.now().isoformat(),
    "projects": PROJECTS,

    "slides": {
        "total_available": len(matched_slides),
        "downloaded": len(downloaded_slides),
        "max_slides_setting": MAX_SLIDES,
        "directory": f"{DATA_DIR}/raw/slides",
    },

    "rnaseq": {
        "total_available": len(matched_rnaseq),
        "downloaded": len(downloaded_rnaseq),
        "directory": f"{DATA_DIR}/raw/rnaseq",
    },

    "matched_patients": len(matched_patients),

    "labels": {
        "bagaev_tme": {
            "source": "https://github.com/BostonGene/MFP",
            "files": list(files.keys()),
            "directory": f"{DATA_DIR}/labels/bagaev_tme",
        },
        "thorsson_panimmune": {
            "source": "https://gdc.cancer.gov/about-data/publications/panimmune",
            "files": list(THORSSON_FILES.keys()),
            "directory": f"{DATA_DIR}/labels/thorsson_panimmune",
        },
        "saltz_til": {
            "source": "doi.org/10.7937/K9/TCIA.2018.Y75F9W1",
            "directory": f"{DATA_DIR}/labels/saltz_til",
        },
        "cbioportal": {
            "source": "datahub.assets.cbioportal.org",
            "studies": list(CBIOPORTAL_STUDIES.keys()),
            "directory": f"{DATA_DIR}/labels/cbioportal",
        },
    },
}

report_path = f"{PROJECT_DIR}/results/phase1_download_report.json"
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)

print("=" * 60)
print("PHASE 1 — DOWNLOAD SUMMARY")
print("=" * 60)
print(f"\nSlides:    {len(downloaded_slides)}/{len(matched_slides)} downloaded (MAX_SLIDES={MAX_SLIDES})")
print(f"RNA-seq:   {len(downloaded_rnaseq)}/{len(matched_rnaseq)} downloaded")
print(f"Matched:   {len(matched_patients)} patients with both slide + RNA-seq")
print(f"\nLabels downloaded:")
print(f"  ✅ Bagaev TME subtypes (BostonGene MFP)")
print(f"  ✅ Thorsson panimmune data (GDC)")
print(f"  ✅ cBioPortal LUAD/LUSC PanCancer Atlas")
print(f"  ℹ️  Saltz TIL maps (check download status above)")
print(f"\nReport: {report_path}")

print(f"\n{'=' * 60}")
print("📋 UPDATE PHASE_TRACKER.md:")
print(f"{'=' * 60}")
print(f"  Status:               DONE")
print(f"  TCGA Slides:          {len(downloaded_slides)}/{len(matched_slides)}")
print(f"  RNA-seq:              {len(downloaded_rnaseq)}/{len(matched_rnaseq)}")
print(f"  Bagaev TME Labels:    Downloaded")
print(f"  Saltz TIL Labels:     Check status")
print(f"  MSI Labels:           Check status")

PHASE 1 — DOWNLOAD SUMMARY

Slides:    20/950 downloaded (MAX_SLIDES=20)
RNA-seq:   13/950 downloaded
Matched:   950 patients with both slide + RNA-seq

Labels downloaded:
  ✅ Bagaev TME subtypes (BostonGene MFP)
  ✅ Thorsson panimmune data (GDC)
  ✅ cBioPortal LUAD/LUSC PanCancer Atlas
  ℹ️  Saltz TIL maps (check download status above)

Report: /content/drive/MyDrive/ImmunoPath/results/phase1_download_report.json

📋 UPDATE PHASE_TRACKER.md:
  Status:               DONE
  TCGA Slides:          20/950
  RNA-seq:              13/950
  Bagaev TME Labels:    Downloaded
  Saltz TIL Labels:     Check status
  MSI Labels:           Check status


In [21]:
print("📁 Current data directory structure:\n")

for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    indent = "  " * level
    folder = os.path.basename(root)
    n_files = len(files)
    if n_files > 5:
        print(f"{indent}{folder}/ ({n_files} files)")
    elif n_files > 0:
        print(f"{indent}{folder}/")
        for f in files[:5]:
            size = os.path.getsize(os.path.join(root, f))
            if size > 1e9:
                print(f"{indent}  {f} ({size/1e9:.1f} GB)")
            elif size > 1e6:
                print(f"{indent}  {f} ({size/1e6:.1f} MB)")
            else:
                print(f"{indent}  {f} ({size/1e3:.1f} KB)")
    else:
        print(f"{indent}{folder}/ (empty)")

print(f"\n{'=' * 60}")
print("NEXT: Phase 2 — Data Processing")
print(f"{'=' * 60}")
print("1. Extract 512×512 patches from .svs slides")
print("2. Apply Reinhard stain normalization")
print("3. Compute immune signatures from RNA-seq")
print("4. Output: patches/ + immune_signatures.csv")
print("\n⚠️  To download MORE slides, increase MAX_SLIDES in Cell 6 and re-run")
print(f"   Currently: MAX_SLIDES = {MAX_SLIDES}")

📁 Current data directory structure:

data/ (empty)
  raw/
    nsclc_metadata.csv (130.2 KB)
    slides/ (20 files)
    rnaseq/ (948 files)
    manifests/
      slide_metadata.csv (183.2 KB)
      rnaseq_metadata.csv (191.2 KB)
  labels/
    thorsson_panimmune.tar.gz (10.0 MB)
    saltz_til.tar.gz (5.0 MB)
    msi_status.tsv (0.0 KB)
    bagaev_tme/
      annotation.tsv (1.8 MB)
      signatures.tsv (4.6 MB)
      signatures/
        gene_signatures.gmt (2.6 KB)
    saltz_til/
      Adjusted_indices_October_23_2017.tsv (387.4 KB)
      TIL-Pattern-Labels.csv (218.5 KB)
      TILMap_TableS1.xlsx (1.7 MB)
      TILMap_TableS2.xlsx (1.3 MB)
      TILmap_Summary_20171013.tsv (5.7 MB)
      08096f8f-7b56-495a-be45-62d5a56f2ee8/ (empty)
        logs/
          TILMap_TableS1.xlsx.parcel (0.2 KB)
      9f533e19-86c1-4501-a049-22d183cb013c/ (empty)
        logs/
          TIL-Pattern-Labels.csv.parcel (0.2 KB)
      bb07f3ec-da93-4f31-825a-263505b161de/ (empty)
        logs/
          TILmap_Su